## analyse du site pour recuperer les url des articles

In [ ]:
from playwright.async_api import async_playwright
import asyncio, re
from urllib.parse import urlparse

async def diag_articles():
    async with async_playwright() as p:
        browser = await p.chromium.connect_over_cdp("http://localhost:9222")
        context = browser.contexts[0]
        page = await context.new_page()
        await page.goto("https://www.lepoint.fr/politique/", wait_until="domcontentloaded")
        await asyncio.sleep(5)

        # 1. Chercher les liens dont le chemin a plus de 2 segments
        #    et un dernier segment de plus de 25 caractères (typique d'un slug)
        liens = await page.evaluate("""
            () => Array.from(document.querySelectorAll('a[href]'))
                .map(a => ({href: a.href, cls: a.className, parent: a.parentElement ? a.parentElement.className : ''}))
        """)

        articles_probables = []
        for l in liens:
            h = l['href']
            if not h.startswith("https://www.lepoint.fr/"):
                continue
            path = urlparse(h).path
            segments = [s for s in path.split("/") if s]
            if len(segments) < 2:
                continue
            last = segments[-1]
            if len(last) > 25 and not any(x in path for x in
                ("/html/", "/rss", "/tag/", "/auteur/", "/abonnement", "/newsletter")):
                articles_probables.append(l)

        print(f"Liens d'articles probables : {len(articles_probables)}")
        for a in articles_probables[:30]:
            print(f"  HREF   : {a['href']}")
            print(f"  CLASS  : {a['cls'][:80]}")
            print(f"  PARENT : {a['parent'][:80]}")
            print()

        # 2. Compter les <article> et <h3> (souvent les titres)
        n_article = await page.evaluate("document.querySelectorAll('article').length")
        n_h3 = await page.evaluate("document.querySelectorAll('h3').length")
        n_h2 = await page.evaluate("document.querySelectorAll('h2').length")
        print(f"<article> : {n_article}")
        print(f"<h3>      : {n_h3}")
        print(f"<h2>      : {n_h2}")

        # 3. Trouver le conteneur principal de la liste
        containers = await page.evaluate("""
            () => {
                const res = [];
                for (const sel of ['main', '.stories', '.stories-list', '.c-stack',
                                   '[class*="stories"]', '[class*="article-list"]',
                                   '[class*="river"]', '[class*="feed"]']) {
                    const el = document.querySelector(sel);
                    if (el) {
                        const links = el.querySelectorAll('a[href]').length;
                        res.push({sel, links: links, cls: el.className});
                    }
                }
                return res;
            }
        """)
        print("\nConteneurs détectés :")
        for c in containers:
            print(f"  {c['sel']:30s} -> {c['links']} liens, class='{c['cls'][:60]}'")

        await page.close()

await diag_articles()

## scrap le point avec playwright

In [ ]:
# -*- coding: utf-8 -*-
import json, time, random, re, asyncio
from datetime import datetime, timezone
from urllib.parse import urlparse
from bson import ObjectId, json_util
from playwright.async_api import async_playwright

# ============ CONFIG ============
CDP_URL = "http://localhost:9222"
START_URL = "https://www.lepoint.fr/politique/bagnolet-pourquoi-la-majorite-lfi-se-dechire-apres-une-reunion-publique-5G5U54WKQRCGFIEH6SEPFCSUK4/"
OUTPUT = "lepoint_articles.jsonl"
TARGET = 2000
DELAY_MIN = 2.0
DELAY_MAX = 4.0
MAX_CLICKS = 200


# ============ HELPERS ============
def _walk_for_article(obj, depth=0):
    if depth > 6:
        return None
    if isinstance(obj, dict):
        if "content_elements" in obj and isinstance(obj["content_elements"], list):
            return obj
        for v in obj.values():
            r = _walk_for_article(v, depth + 1)
            if r:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = _walk_for_article(v, depth + 1)
            if r:
                return r
    return None


def _clean_html(text):
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("&nbsp;", " ").replace("&amp;", "&")
    text = text.replace("&#8217;", "'").replace("&#039;", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


async def extract_from_fusion(page):
    raw = await page.evaluate("""
        () => {
            try {
                if (window.Fusion && window.Fusion.globalContent) {
                    return JSON.stringify(window.Fusion.globalContent);
                }
            } catch(e) {}
            return null;
        }
    """)
    if not raw:
        return None, None
    try:
        data = json.loads(raw)
    except Exception as e:
        print("  Erreur parsing JSON Fusion :", e)
        return None, None

    art = _walk_for_article(data)
    if not art:
        return None, None

    title = ""
    hl = art.get("headlines", {})
    if isinstance(hl, dict):
        title = hl.get("basic") or hl.get("headline") or hl.get("mobile") or ""
    if not title:
        title = art.get("title") or ""
    title = _clean_html(title)

    paragraphs = []
    for el in art.get("content_elements", []):
        if isinstance(el, dict) and el.get("type") == "text":
            t = _clean_html(el.get("content", ""))
            if t and len(t) > 5:
                paragraphs.append(t)

    return title, " ".join(paragraphs)


async def extract_article(page, url):
    data = {
        "_id": ObjectId(),
        "url": url,
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "content": "",
        "section": "",
        "source_id": "lepoint",
        "title": "",
    }

    try:
        await page.goto(url, timeout=30000, wait_until="domcontentloaded")
        await page.wait_for_selector("h1", timeout=15000)
        await asyncio.sleep(2)
    except Exception as e:
        print(f"  Timeout/erreur sur {url} : {e}")
        return None

    title, content = await extract_from_fusion(page)

    if not content or len(content) < 100:
        try:
            h1 = await page.locator("h1").first.inner_text(timeout=5000)
            if not title:
                title = h1
        except Exception:
            pass
        if not content:
            for sel in ["div.art-text", "div.article__content", "article"]:
                try:
                    cont = page.locator(sel).first
                    if await cont.count() > 0:
                        parts = []
                        for p in await cont.locator("p").all():
                            t = (await p.inner_text()).strip()
                            if t and len(t) > 20:
                                parts.append(t)
                        if parts:
                            content = " ".join(parts)
                            break
                except Exception:
                    continue

    data["title"] = title or ""
    data["content"] = content or ""
    path = urlparse(url).path.strip("/")
    if path:
        data["section"] = path.split("/")[0]

    if len(data["content"]) < 100:
        print(f"  Contenu trop court, ignoré : {url}")
        return None
    return data


def is_article_url(url):
    """Filtre les URLs d'articles Le Point, en excluant tout ce qui est navigation."""
    if not url.startswith("https://www.lepoint.fr/"):
        return False
    path = urlparse(url).path

    # Exclusions strictes (navigation, rubriques, pages spéciales)
    for bad in ("/html/", "/rss", "/sitemap", "/abonnement", "/newsletter",
                "/tag/", "/tags/", "/auteur/", "/recherche", "/contact",
                "/mentions", "/politique-de-protection", "/cookies",
                "/cgu", "/cgv", "/videos/", "/podcasts/", "/photos/",
                "/magazine/", "/connexion", "/moncompte", "/postillon",
                "/editos-du-point", "/debats", "/stories/", "/afrique/",
                "/medias/", "/services/", "/jeux-concours"):
        if bad in path:
            return False

    segments = [s for s in path.split("/") if s]
    if len(segments) < 2:
        return False

    last = segments[-1]
    # Un slug d'article fait au moins 25 caractères
    if len(last) < 25:
        return False

    # Un slug contient au moins 4 tirets (titre avec plusieurs mots)
    if last.count("-") < 3:
        return False

    return True


async def collect_urls_from_page(page):
    """Extrait les URLs d'articles depuis le conteneur .stories-list uniquement."""
    links = await page.evaluate("""
        () => {
            const root = document.querySelector('.stories-list')
                      || document.querySelector('.stories')
                      || document.querySelector('main')
                      || document;
            return Array.from(root.querySelectorAll('a[href]')).map(a => a.href);
        }
    """)
    urls = set()
    for u in links:
        if is_article_url(u):
            urls.add(u)
    return urls

# Liste élargie de rubriques Le Point
SECTIONS = [
    "politique", "international", "economie", "societe",
    "culture", "sports", "sciences", "high-tech-internet",
    "medias", "monde", "france", "justice", "faits-divers",
    "sante", "environnement", "immobilier", "argent", "education",
    "debats", "editos-du-point", "afrique", "postillon",
]

async def collect_all_urls(page, target=2000):
    """Collecte jusqu'à 2000 URLs en parcourant toutes les rubriques."""
    print("=== Collecte multi-rubriques Le Point ===")
    all_urls = set()

    for section in SECTIONS:
        if len(all_urls) >= target:
            break

        section_url = f"https://www.lepoint.fr/{section}/"
        print(f"\n--- {section} ---")
        try:
            await page.goto(section_url, wait_until="domcontentloaded", timeout=30000)
            await asyncio.sleep(4)
        except Exception as e:
            print(f"  Erreur chargement : {e}")
            continue

        section_urls = await collect_urls_from_page(page)
        before = len(all_urls)
        all_urls |= section_urls
        print(f"  Page initiale : +{len(all_urls) - before} URLs (total {len(all_urls)})")

        clicks = 0
        failures = 0
        while clicks < MAX_CLICKS and len(all_urls) < target:
            btn = await page.query_selector("button.stories-see-more")
            if not btn or not await btn.is_visible():
                print(f"  Plus de bouton après {clicks} clics.")
                break
            try:
                await btn.scroll_into_view_if_needed()
                await asyncio.sleep(1)
                await btn.click()
                clicks += 1
            except Exception:
                failures += 1
                if failures >= 3:
                    break
                await asyncio.sleep(2)
                continue

            # Attente active : max 8s pour voir de nouveaux liens
            prev = len(all_urls)
            for _ in range(8):
                await asyncio.sleep(1)
                new_urls = await collect_urls_from_page(page)
                if len(new_urls | all_urls) > prev:
                    break

            new_urls = await collect_urls_from_page(page)
            added = new_urls - all_urls
            if added:
                all_urls.update(new_urls)
                failures = 0
                print(f"  Clic {clicks} : +{len(added)} (total {len(all_urls)})")
            else:
                failures += 1
                if failures >= 3:
                    break

        print(f"  Cumul : {len(all_urls)} URLs")

    urls_list = list(all_urls)[:target]
    print(f"\n>>> {len(urls_list)} URLs uniques collectées pour Le Point.")
    return urls_list

# ============ MAIN ASYNC ============
async def main_async():
    async with async_playwright() as p:
        browser = await p.chromium.connect_over_cdp(CDP_URL)
        context = browser.contexts[0]
        page = await context.new_page()

        # --- Test sur l'article fourni ---
        print(f"Test sur : {START_URL}")
        try:
            await page.goto(START_URL, wait_until="domcontentloaded", timeout=30000)
            await page.wait_for_selector("h1", timeout=15000)
            await asyncio.sleep(2)
            title, content = await extract_from_fusion(page)
            print(f"  Titre  : {title}")
            print(f"  Longueur contenu : {len(content) if content else 0}")
            if content:
                print(f"  Extrait : {content[:200]}...")
        except Exception as e:
            print(f"  Erreur test : {e}")
            return

        # --- Collecte des URLs ---
        urls = await collect_all_urls(page, TARGET)
        if not urls:
            print("Aucune URL trouvée.")
            return

        # --- Extraction de tous les articles ---
        ok = ko = 0
        with open(OUTPUT, "w", encoding="utf-8") as f:
            for i, url in enumerate(urls, 1):
                print(f"[{i}/{len(urls)}] {url}")
                data = await extract_article(page, url)
                if data:
                    f.write(json_util.dumps(data, ensure_ascii=False) + "\n")
                    f.flush()
                    ok += 1
                else:
                    ko += 1
                await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        print(f"\nTerminé. {ok} articles sauvegardés, {ko} ignorés.")
        print(f"Fichier : {OUTPUT}")
        await page.close()


# ============ LANCEMENT DANS JUPYTER ============
# En Jupyter, on utilise 'await' directement, PAS asyncio.run()
await main_async()

## scrap gamespot avec playwright

In [ ]:
# -*- coding: utf-8 -*-
import json, time, random, re, asyncio
from datetime import datetime, timezone
from urllib.parse import urlparse
from playwright.async_api import async_playwright

# ============ CONFIG ============
CDP_URL = "http://localhost:9222"
OUTPUT_GS = "gamespot_articles.jsonl"
TARGET_GS = 800
DELAY_MIN = 2.0
DELAY_MAX = 4.0
MAX_LOAD_MORE = 80

GAMESPOT_CATEGORIES = [
    "https://www.gamespot.com/news/",
    "https://www.gamespot.com/reviews/",
    "https://www.gamespot.com/articles/",
    "https://www.gamespot.com/guides/",
    "https://www.gamespot.com/features/",
]

GAMESPOT_RSS = [
    "https://www.gamespot.com/feeds/mashup/",
    "https://www.gamespot.com/feeds/game-news/",
    "https://www.gamespot.com/feeds/reviews/",
]


# ============ FILTRAGE ============
def is_gamespot_article(url):
    if "gamespot.com" not in url:
        return False
    if "#" in url:
        return False

    path = urlparse(url).path

    # Exclure les commentaires / pagination interne (ex: /1100-XXXX/22/)
    if re.search(r"/\d{5,}/\d+/?$", path):
        return False

    # Exclure les pages non-articles
    for bad in ("/videos/", "/galleries/", "/forums/", "/user/",
                "/deals/", "/newsletters/", "/about/", "/contact/",
                "/privacy", "/terms", "/wp-content/", "/author/",
                "/tag/", "/search"):
        if bad in path:
            return False

    # Articles GameSpot
    if re.search(r"/(articles|reviews|guides|features|news)/[^/]+/?$", path):
        segments = [s for s in path.split("/") if s]
        if len(segments) >= 2 and len(segments[-1]) > 5:
            return True
    return False


def clean_html(text):
    if not text:
        return ""
    if isinstance(text, list):
        text = " ".join(str(t) for t in text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("&nbsp;", " ").replace("&amp;", "&")
    text = text.replace("&#8217;", "'").replace("&#039;", "'")
    text = text.replace("&quot;", '"').replace("&lt;", "<").replace("&gt;", ">")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_section_from_jsonld(section):
    """Gère les cas où articleSection est une liste, str, ou absent."""
    if isinstance(section, list):
        section = section[0] if section else ""
    if not isinstance(section, str):
        return ""
    return section.lower().replace(" ", "-")


# ============ COLLECTE URLs ============
async def collect_from_rss(page):
    urls = set()
    for feed in GAMESPOT_RSS:
        try:
            await page.goto(feed, wait_until="domcontentloaded", timeout=20000)
            await asyncio.sleep(2)
            links = await page.evaluate("""
                () => Array.from(document.querySelectorAll('item link, entry link'))
                    .map(el => el.textContent.trim())
            """)
            for u in links:
                if is_gamespot_article(u):
                    urls.add(u)
        except Exception:
            pass
    return urls


async def collect_from_category_with_load_more(page, cat_url, current_total, target, max_clicks=80):
    """Collecte les URLs d'une catégorie, s'arrête dès que le total global atteint la cible."""
    urls = set()
    try:
        await page.goto(cat_url, wait_until="domcontentloaded", timeout=30000)
        await asyncio.sleep(3)
    except Exception as e:
        print(f"    Erreur chargement {cat_url} : {e}")
        return urls

    # URLs initiales
    links = await page.evaluate("() => Array.from(document.querySelectorAll('a[href]')).map(a => a.href)")
    for u in links:
        if is_gamespot_article(u):
            urls.add(u)
    print(f"    Page initiale : {len(urls)} URLs")

    if current_total + len(urls) >= target:
        print(f"    Cible atteinte dès la page initiale.")
        return urls

    clicks = 0
    failures = 0
    while clicks < max_clicks:
        # ARRÊT ANTICIPÉ si on atteint la cible globale
        if current_total + len(urls) >= target:
            print(f"    Cible globale atteinte ({current_total + len(urls)} >= {target}). Arrêt.")
            break

        btn = await page.query_selector("button#js-archive-load-more")
        if not btn or not await btn.is_visible():
            print(f"    Plus de bouton après {clicks} clics.")
            break

        try:
            await btn.scroll_into_view_if_needed()
            await asyncio.sleep(0.5)
            await btn.click()
            clicks += 1
        except Exception as e:
            failures += 1
            if failures >= 3:
                break
            await asyncio.sleep(2)
            continue

        # Attente que de nouveaux articles apparaissent
        prev = len(urls)
        for _ in range(8):
            await asyncio.sleep(0.7)
            links = await page.evaluate("() => Array.from(document.querySelectorAll('a[href]')).map(a => a.href)")
            for u in links:
                if is_gamespot_article(u):
                    urls.add(u)
            if len(urls) > prev:
                break

        added = len(urls) - prev
        if added > 0:
            failures = 0
            print(f"    Clic {clicks} : +{added} (total {len(urls)}, global {current_total + len(urls)})")
        else:
            failures += 1
            if failures >= 3:
                break

    return urls

async def collect_all_urls(page, target=800):
    print("=== Collecte URLs GameSpot ===")
    all_urls = set()

    # 1. RSS
    print("\n--- RSS ---")
    rss_urls = await collect_from_rss(page)
    all_urls |= rss_urls
    print(f"  RSS : {len(all_urls)} URLs")

    if len(all_urls) >= target:
        return list(all_urls)[:target]

    # 2. Catégories avec Load More (arrêt dès que target atteinte)
    print("\n--- Catégories (Load More) ---")
    for cat in GAMESPOT_CATEGORIES:
        if len(all_urls) >= target:
            print(f"\n>>> Cible atteinte, on arrête la collecte.")
            break
        print(f"\n  {cat}")
        try:
            cat_urls = await collect_from_category_with_load_more(
                page, cat, len(all_urls), target, max_clicks=MAX_LOAD_MORE
            )
            all_urls |= cat_urls
            print(f"    Cumul global : {len(all_urls)}")
        except Exception as e:
            print(f"    Erreur : {e}")

    urls_list = list(all_urls)[:target]
    print(f"\n>>> {len(urls_list)} URLs GameSpot collectées.")
    return urls_list

# ============ EXTRACTION ============
async def extract_article(page, url):
    data = {
        "_id": f"gamespot_{abs(hash(url)) % 10**12}",
        "url": url,
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "content": "",
        "section": "",
        "source_id": "gamespot",
        "title": "",
    }

    try:
        # 1. Navigation avec attente souple
        await page.goto(url, wait_until="domcontentloaded", timeout=45000)
        # 2. Attente courte et non-bloquante pour le titre
        try:
            await page.wait_for_selector("h1", timeout=8000)
        except Exception:
            # Si pas de h1 après 8s, on continue quand même
            pass
        # 3. Pause fixe pour laisser le contenu se charger
        await asyncio.sleep(3)
    except Exception as e:
        print(f"  Timeout navigation : {url}")
        return None

    # --- Titre via <title> ou og:title (toujours présent) ---
    title = await page.evaluate("""
        () => {
            const og = document.querySelector('meta[property="og:title"]');
            if (og) return og.getAttribute('content');
            const t = document.querySelector('title');
            return t ? t.textContent.replace(/ - GameSpot$/, '').trim() : '';
        }
    """)
    data["title"] = clean_html(title)

    # --- Contenu via JSON-LD ---
    jsonld_raw = await page.evaluate("""
        () => {
            const scripts = document.querySelectorAll('script[type="application/ld+json"]');
            for (const s of scripts) {
                try {
                    const obj = JSON.parse(s.textContent);
                    const items = Array.isArray(obj) ? obj : (obj['@graph'] || [obj]);
                    for (const item of items) {
                        if (!item) continue;
                        const t = item['@type'];
                        const types = Array.isArray(t) ? t : [t];
                        if (types.some(x => ['NewsArticle', 'Article', 'Review',
                                             'BlogPosting', 'WebPage'].includes(x))) {
                            return JSON.stringify(item);
                        }
                    }
                } catch(e) {}
            }
            return null;
        }
    """)

    if jsonld_raw:
        try:
            obj = json.loads(jsonld_raw)
            if not data["title"]:
                data["title"] = clean_html(obj.get("headline", ""))
            body = obj.get("articleBody", "") or obj.get("reviewBody", "")
            data["content"] = clean_html(body)
            data["section"] = get_section_from_jsonld(obj.get("articleSection", ""))
        except Exception as e:
            print(f"  Erreur JSON-LD : {e}")

    # --- Fallback HTML ---
    if not data["content"] or len(data["content"]) < 100:
        try:
            for sel in ["div.js-content-owner", "div.article-content",
                        "div[itemprop='articleBody']",
                        "div[data-component='content']",
                        "div.wp-block-post-content",
                        "article"]:
                cont = page.locator(sel).first
                if await cont.count() > 0:
                    parts = []
                    for p in await cont.locator("p").all():
                        t = (await p.inner_text()).strip()
                        if t and len(t) > 20:
                            parts.append(t)
                    if parts:
                        data["content"] = " ".join(parts)
                        break
        except Exception:
            pass

    if not data["section"]:
        path = urlparse(url).path.strip("/")
        if path:
            data["section"] = path.split("/")[0]

    if len(data["content"]) < 100:
        print(f"  Contenu trop court, ignoré : {url}")
        return None

    return data

# ============ MAIN ============
async def scrape_gamespot():
    async with async_playwright() as p:
        browser = await p.chromium.connect_over_cdp(CDP_URL)
        context = browser.contexts[0]
        page = await context.new_page()

        urls = await collect_all_urls(page, TARGET_GS)
        if not urls:
            print("Aucune URL GameSpot.")
            return

        ok = ko = 0
        with open(OUTPUT_GS, "w", encoding="utf-8") as f:
            for i, url in enumerate(urls, 1):
                print(f"[{i}/{len(urls)}] {url}")
                data = await extract_article(page, url)
                if data:
                    f.write(json.dumps(data, ensure_ascii=False) + "\n")
                    f.flush()
                    ok += 1
                else:
                    ko += 1
                await asyncio.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        print(f"\nTerminé GameSpot. {ok} sauvegardés, {ko} ignorés.")
        print(f"Fichier : {OUTPUT_GS}")
        await page.close()


await scrape_gamespot()

## importation vers mongo

In [ ]:
# -*- coding: utf-8 -*-
import json, os
from datetime import datetime, timezone
from pymongo import MongoClient, ASCENDING, UpdateOne
from pymongo.errors import BulkWriteError

# ==== CONFIG ====
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "tx_analyzer"
COLLECTION = "articles"
FILE = "gamespot_articles.jsonl"
BATCH_SIZE = 500


def flush_batch(collection, batch):
    ops = []
    for doc in batch:
        ops.append(UpdateOne({"url": doc["url"]}, {"$set": doc}, upsert=True))
    try:
        r = collection.bulk_write(ops, ordered=False)
        return r.upserted_count, r.modified_count
    except BulkWriteError as e:
        print(f"  BulkWriteError : {e.details.get('writeErrors', [])[:2]}")
        return 0, 0


def import_jsonl(collection, filepath):
    if not os.path.exists(filepath):
        print(f"Fichier introuvable : {filepath}")
        return 0, 0

    inserted = updated = errors = 0
    batch = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                doc = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"  Ligne {line_num} invalide : {e}")
                errors += 1
                continue

            # Normaliser _id
            if "_id" in doc and not isinstance(doc["_id"], (str, int)):
                doc["_id"] = str(doc["_id"])

            # Enrichissement
            doc["char_count"] = len(doc.get("content", ""))
            doc["word_count"] = len(doc.get("content", "").split())
            doc["imported_at"] = datetime.now(timezone.utc)

            batch.append(doc)
            if len(batch) >= BATCH_SIZE:
                ins, upd = flush_batch(collection, batch)
                inserted += ins
                updated += upd
                batch = []

    if batch:
        ins, upd = flush_batch(collection, batch)
        inserted += ins
        updated += upd

    return inserted, updated


def main():
    client = MongoClient(MONGO_URI)
    db = client[DB_NAME]
    collection = db[COLLECTION]

    # Index
    try:
        collection.create_index([("url", ASCENDING)], unique=True)
        collection.create_index([("source_id", ASCENDING)])
        collection.create_index([("section", ASCENDING)])
        collection.create_index([("collected_at", ASCENDING)])
        print("Index vérifiés : url (unique), source_id, section, collected_at.")
    except Exception as e:
        print(f"Note index : {e}")

    print(f"\n=== Import de {FILE} dans {DB_NAME}.{COLLECTION} ===")
    ins, upd = import_jsonl(collection, FILE)
    print(f"  {ins} insérés, {upd} mis à jour")

    # Résumé par source
    print(f"\n=== Résumé ===")
    total = collection.count_documents({})
    print(f"Documents totaux dans {DB_NAME}.{COLLECTION} : {total}")
    for src in ["lepoint", "gamespot"]:
        n = collection.count_documents({"source_id": src})
        print(f"  source_id='{src}' : {n}")

    client.close()


main()